# 11.2 · 词袋与 TF-IDF / Bag-of-Words & TF-IDF

> **课程定位 / Where this fits**
> 第 2 课，**Part 11 · 经典 NLP**。
> Lesson 2, **Part 11 · Classic NLP**.
>
> 11.1 把文本清洗成了 token。但模型只懂**数字**——怎么把一篇文档变成一个**定长数字向量**？最经典的两种方法：**词袋(BoW)** 数每个词出现几次；**TF-IDF** 在此基础上给"既在本文常见、又在全语料稀有"的词更高权重(突出关键词)。它们简单、可解释、至今仍是文本分类/检索的强基线。本课**从零实现 BoW 和 TF-IDF**并与 sklearn 对照，配**矩阵热图、关键词可视化**。
> Lesson 11.1 cleaned text into tokens. But models understand only **numbers** — how to turn a document into a **fixed-length numeric vector**? The two classics: **Bag-of-Words (BoW)** counts each word; **TF-IDF** additionally upweights words that are frequent *in this doc* yet rare *across the corpus* (highlighting keywords). Simple, interpretable, still strong baselines for text classification/retrieval. We **implement BoW and TF-IDF from scratch** vs sklearn, with **matrix heatmaps and keyword visualizations**.
>
> 💼 **实战/面试视角**："TF-IDF 公式与直觉 / 为什么不用纯词频 / 稀疏性 / n-gram" 是 NLP 必考。
> 💼 **Practical/interview angle:** "TF-IDF formula & intuition / why not raw counts / sparsity / n-grams" — NLP essentials.

> 📐 **符号约定 / Notation**
> - $\text{tf}(t,d)$ —— 词 $t$ 在文档 $d$ 的出现次数 / term frequency
> - $\text{df}(t)$ —— 包含词 $t$ 的文档数 / document frequency
> - $\text{idf}(t)=\log\frac{N}{\text{df}(t)}$ —— 逆文档频率 / inverse document frequency

> 💡 **面试相关 / Interview-relevant**
> - "TF-IDF 的公式和每一项的直觉"（出镜率 ★★★★★）
> - "为什么用 IDF / 纯词频有什么问题"（★★★★★）
> - "文本向量为什么稀疏、维度多高"（★★★★）
> - "n-gram 解决什么(词序/否定)"（★★★★）
> - "BoW/TF-IDF 的局限(无语义/词序)"（★★★★）

---

## 学习目标 / Learning Objectives
1. 理解为什么要把文档变成定长向量(one-hot→BoW)。
   Understand why documents become fixed-length vectors (one-hot→BoW).
2. **从零实现词袋(BoW)** 与文档-词矩阵。
   Implement Bag-of-Words and the document-term matrix from scratch.
3. **从零实现 TF-IDF**，理解 TF×IDF 的直觉。
   Implement TF-IDF from scratch; grasp the TF×IDF intuition.
4. 用 n-gram 捕捉词序/否定，理解维度与稀疏。
   Use n-grams for word order/negation; understand dimensionality and sparsity.
5. 知道 BoW/TF-IDF 的局限(无语义、无词序)。
   Know the limits of BoW/TF-IDF (no semantics, no order).

## 目录 / TOC
1. [从 token 到向量：one-hot ⭐](#1)
2. [词袋 BoW（从零 + sklearn）⭐](#2)
3. [TF-IDF（从零 + 直觉）⭐](#3)
4. [n-gram、稀疏性与局限 + 小结 ⭐](#4)


<a id="1"></a>
## 1. 从 token 到向量：one-hot ⭐ / From Tokens to Vectors: One-Hot

模型要数字向量，且**每篇文档的向量必须等长**(才能堆成矩阵喂给模型)。但文档长短不一、用词各异，怎么对齐？
Models need numeric vectors, and **every document's vector must be the same length** (to stack into a matrix). But documents differ in length and words — how to align?

第一步：建一个**词表(vocabulary)**——把全语料所有不同的词编号(词1→0号位, 词2→1号位…)。这样每个词对应向量里一个固定位置。最朴素的表示是 **one-hot**：一个词 = 一个"只有它那一位是 1、其余全 0"的向量。
First: build a **vocabulary** — number every unique word in the corpus (word1→slot 0, word2→slot 1…). Each word maps to a fixed position. The simplest representation is **one-hot**: a word = a vector that's 1 at its slot and 0 elsewhere.


In [ ]:
import numpy as np, matplotlib.pyplot as plt, seaborn as sns
sns.set_theme(style="white")

corpus = [                                                # 4 篇迷你"文档" / 4 tiny documents
    "the cat sat on the mat",
    "the dog sat on the log",
    "cats and dogs are friends",
    "the cat and the dog play",
]
# 建词表: 收集所有不同的词并编号 / build vocabulary (unique words → indices)
vocab = sorted(set(w for doc in corpus for w in doc.split()))
word2idx = {w: i for i, w in enumerate(vocab)}
print(f"词表({len(vocab)}个词): {vocab}")
# one-hot: "cat" → 只有 cat 那一位是1 / one-hot for "cat"
oh = np.zeros(len(vocab)); oh[word2idx["cat"]] = 1
print(f"\n'cat' 的 one-hot 向量: {oh.astype(int)}")
print(f"长度 = 词表大小 = {len(vocab)}; 只有第 {word2idx['cat']} 位(cat)是1")
print("问题: one-hot 每个词彼此正交(cat 和 dog 距离 = cat 和 the 距离), 不含任何语义; 而且一篇文档有很多词")


<a id="2"></a>
## 2. 词袋 BoW（从零 + sklearn）⭐ / Bag-of-Words

**词袋(Bag-of-Words)**：把一篇文档表示成"**每个词出现了几次**"的向量——长度=词表大小，第 $i$ 位 = 词表第 $i$ 个词在该文档的出现次数。相当于把文档里所有词的 one-hot **加起来**。
**Bag-of-Words:** represent a document as "**how many times each word appears**" — length = vocabulary size, slot $i$ = count of vocab word $i$ in the document. It's the **sum** of the one-hot vectors of all words in the doc.

"袋"的含义：**只记词频，丢掉词序**——"dog bites man"和"man bites dog"的 BoW 完全一样。这是 BoW 的核心简化(也是局限)。
"Bag" means: **only counts, order discarded** — "dog bites man" and "man bites dog" have identical BoW. The core simplification (and limitation).

把所有文档的 BoW 堆起来 = **文档-词矩阵(document-term matrix)**：行=文档，列=词。
Stacking all docs' BoW = the **document-term matrix**: rows=documents, columns=words.


In [ ]:
def bow_matrix(corpus, vocab):
    """从零构建文档-词矩阵 / build document-term matrix from scratch."""
    w2i = {w: i for i, w in enumerate(vocab)}
    M = np.zeros((len(corpus), len(vocab)), dtype=int)
    for d, doc in enumerate(corpus):
        for w in doc.split():
            if w in w2i: M[d, w2i[w]] += 1               # 该词计数 +1 / increment that word's count
    return M

M = bow_matrix(corpus, vocab)
fig, ax = plt.subplots(figsize=(11, 3.2))
sns.heatmap(M, annot=True, fmt="d", xticklabels=vocab, yticklabels=[f"doc{i}" for i in range(len(corpus))],
            cmap="Blues", cbar_kws={"label":"出现次数"}, ax=ax)
ax.set_title("文档-词矩阵 (BoW): 行=文档, 列=词, 值=词频"); plt.tight_layout(); plt.show()
print("每行就是一篇文档的 BoW 向量; 'the' 在 doc0 出现2次 → 对应位置=2")
print("注意: BoW 丢词序('cat sat'和'sat cat'一样); 高频词(the)数值大但信息少 → TF-IDF 来修正")

# sklearn 一行搞定(与我们一致) / sklearn does it in one line
from sklearn.feature_extraction.text import CountVectorizer
cv = CountVectorizer()
Msk = cv.fit_transform(corpus)                           # 返回稀疏矩阵 / sparse matrix
print(f"\nsklearn CountVectorizer: 矩阵形状 {Msk.shape}, 词表 {list(cv.get_feature_names_out())}")
print("实战用 sklearn(稀疏存储, 高效); 原理就是上面的计数")


<a id="3"></a>
## 3. TF-IDF（从零 + 直觉）⭐ / TF-IDF From Scratch & Intuition

BoW 有个大问题：**高频词(the, is)数值很大，却几乎没有区分文档的能力**。我们想要的是**能代表一篇文档独特主题的词**。
BoW has a big flaw: **frequent words (the, is) get large counts but barely distinguish documents**. We want **words that characterize a document's unique topic**.

**TF-IDF** = **TF（词频）× IDF（逆文档频率）**，两项相乘：
**TF-IDF** = **TF (term frequency) × IDF (inverse document frequency)**:
- **TF**：词在**本文档**出现越多越重要 → `tf(t,d)` = 出现次数(或其归一化)。
  **TF:** the more a word appears **in this document**, the more important → `tf(t,d)` = count (or normalized).
- **IDF**：词在**越少文档**里出现越独特/有信息 → `idf(t) = log(N / df(t))`，$N$=总文档数，$df$=含该词的文档数。出现在所有文档里的词(the)→ idf≈0(被压低)；只在少数文档出现的词(rocket)→ idf 大(被突出)。
  **IDF:** the **fewer documents** a word appears in, the more distinctive → `idf(t) = log(N / df(t))`. A word in every doc (the) → idf≈0 (suppressed); a word in few docs (rocket) → large idf (highlighted).

**直觉一句话**：TF-IDF 高 = "在这篇里常出现，但在别处罕见" = **这篇文档的关键词**。
**One-liner:** high TF-IDF = "frequent here but rare elsewhere" = **a keyword of this document**.


In [ ]:
def tfidf_from_scratch(corpus, vocab):
    w2i = {w: i for i, w in enumerate(vocab)}
    N = len(corpus)
    tf = bow_matrix(corpus, vocab).astype(float)         # 词频矩阵 / term-frequency matrix
    df = (tf > 0).sum(axis=0)                             # 每个词出现在多少篇文档 / document frequency
    idf = np.log(N / df)                                  # 逆文档频率 / inverse document frequency
    return tf * idf, idf                                  # 逐元素 TF×IDF / elementwise TF*IDF

tfidf, idf = tfidf_from_scratch(corpus, vocab)
# 展示几个词的 IDF: 越普遍越低 / show IDF: common words low
idf_sorted = sorted(zip(vocab, idf), key=lambda x: x[1])
print("IDF 从低到高(越低=越普遍=越没区分度):")
for w, v in idf_sorted[:3]: print(f"  {w:<8} idf={v:.2f}  (出现在很多文档→被压低)")
for w, v in idf_sorted[-3:]: print(f"  {w:<8} idf={v:.2f}  (只在少数文档→被突出)")

fig, ax = plt.subplots(figsize=(11, 3.2))
sns.heatmap(tfidf, annot=True, fmt=".1f", xticklabels=vocab, yticklabels=[f"doc{i}" for i in range(len(corpus))],
            cmap="OrRd", cbar_kws={"label":"TF-IDF"}, ax=ax)
ax.set_title("TF-IDF 矩阵: 高频普遍词(the)被压低, 文档独特词被突出"); plt.tight_layout(); plt.show()
print("\n对比 BoW: 'the' 在 BoW 里数值最大, 在 TF-IDF 里却≈0(因为它出现在所有文档, idf≈0)")
print("TF-IDF 高的词 = 该文档的关键词(本文常见+全局稀有)")


In [ ]:
# 在真实语料上看 TF-IDF 提取的"关键词" / TF-IDF keywords on a real corpus
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import TfidfVectorizer
data = fetch_20newsgroups(subset="train", categories=["sci.space","rec.sport.baseball"],
                          remove=("headers","footers","quotes"))
tfv = TfidfVectorizer(stop_words="english", min_df=5, max_df=0.5)   # 去停用词+过滤极端词 / filter extremes
X = tfv.fit_transform(data.data)                          # 稀疏 TF-IDF 矩阵 / sparse TF-IDF matrix
terms = np.array(tfv.get_feature_names_out())
print(f"TF-IDF 矩阵形状: {X.shape}  (文档数 × 词表大小)")
print(f"稀疏度: 非零元素仅占 {X.nnz/(X.shape[0]*X.shape[1])*100:.2f}%  ← 文本向量极度稀疏(Zipf长尾)")
# 每篇文档 TF-IDF 最高的词 = 关键词 / top TF-IDF terms per document = keywords
fig, axes = plt.subplots(1, 3, figsize=(13, 3.4))
for ax, di in zip(axes, [0, 1, 5]):
    row = X[di].toarray().ravel()
    top = row.argsort()[-8:]                              # 该文档 TF-IDF 最高的8个词 / top-8 terms
    ax.barh(terms[top], row[top], color="#39c")
    ax.set_title(f"doc{di} ({data.target_names[data.target[di]]}) 关键词", fontsize=9)
plt.tight_layout(); plt.show()
print("TF-IDF 自动抽出每篇文档的主题词(空间帖→orbit/space, 棒球帖→game/team) → 可解释且实用")
print("注: sklearn 的 TF-IDF 有平滑(idf+1)和 L2 归一化, 数值与纯公式略不同, 思想一致")


<a id="4"></a>
## 4. n-gram、稀疏性与局限 + 小结 ⭐ / n-grams, Sparsity & Limits

**n-gram**：BoW 丢了词序，但词序有时关键——`"not good"` 和 `"good"` 意思相反！**n-gram** 把**连续 n 个词**当作一个 token：**bigram(2-gram)** 会把 `not_good` 作为一个特征，从而捕捉否定/搭配。
**n-grams:** BoW loses order, but order matters — `"not good"` vs `"good"` are opposite! An **n-gram** treats **n consecutive words** as one token: a **bigram** captures `not_good` as a feature, recovering negation/collocations.

代价：**维度爆炸 + 更稀疏**。unigram 词表可能几万，加上 bigram 可能上百万。所以实战会用 `min_df`/`max_features` 控制。
Cost: **dimensionality explosion + more sparsity.** Unigram vocab may be tens of thousands; adding bigrams → millions. So we cap with `min_df`/`max_features`.

**BoW/TF-IDF 的根本局限(面试必答)**：
**Fundamental limits of BoW/TF-IDF (must-know):**
- **无语义**：`"car"` 和 `"automobile"` 是两个完全无关的维度(正交)，模型不知道它们近义。
  **No semantics:** `"car"` and `"automobile"` are unrelated (orthogonal) dimensions; the model doesn't know they're synonyms.
- **(基本)无词序**：靠 n-gram 只能局部缓解。
  **(Mostly) no word order:** n-grams only partially help.
- **高维稀疏**：词表多大维度就多大。
  **High-dimensional & sparse:** dimension = vocabulary size.

→ 这些正是 **11.3 词向量(Word2Vec)** 要解决的：用**低维稠密**向量编码**语义**。
→ These motivate **11.3 Word Embeddings (Word2Vec)**: encode **semantics** with **low-dimensional dense** vectors.


In [ ]:
# bigram 捕捉否定: "not good" vs "good" / bigrams capture negation
texts = ["the movie is good", "the movie is not good"]
uni = CountVectorizer(ngram_range=(1,1)).fit(texts)
bi  = CountVectorizer(ngram_range=(1,2)).fit(texts)       # 含 unigram + bigram / unigrams + bigrams
print("unigram 特征:", list(uni.get_feature_names_out()))
print("  → 两句都含 good, 模型难区分正负面")
print("bigram 特征 :", list(bi.get_feature_names_out()))
print("  → 出现 'not good' 这个特征 → 能捕捉否定!\n")

# 维度对比: 加 bigram 维度暴涨 / dimensionality blow-up with bigrams
print("不做频次过滤时(min_df=1):")
for ng, name in [((1,1),"unigram"), ((1,2),"uni+bigram")]:
    v = TfidfVectorizer(stop_words="english", min_df=1, ngram_range=ng).fit(data.data)
    print(f"  {name:<12} 词表维度 = {len(v.get_feature_names_out()):,}")
print("→ 加 bigram 维度暴涨(绝大多数 bigram 只出现一两次→长尾)")
print("加 min_df=5 过滤稀有词后:")
for ng, name in [((1,1),"unigram"), ((1,2),"uni+bigram")]:
    v = TfidfVectorizer(stop_words="english", min_df=5, ngram_range=ng).fit(data.data)
    print(f"  {name:<12} 词表维度 = {len(v.get_feature_names_out()):,}")
print("→ min_df/max_features 能砍掉大部分稀有 bigram, 控制维度(实战必用)")


```
token→向量: 先建词表; one-hot(一个词一位); 但词彼此正交无语义
BoW(词袋): 文档=词频向量, 丢词序; 文档-词矩阵(行文档列词); 高频词数值大但没区分度
TF-IDF = TF(本文词频) × IDF(log(N/df), 越普遍越低); 高=本文常见+全局稀有=关键词
n-gram: 连续n词当一个token, bigram捕捉'not good'否定; 代价维度暴涨更稀疏(min_df控制)
稀疏: 文本向量非零元极少(Zipf长尾); 维度=词表大小
局限: 无语义(car≠automobile正交) + 基本无词序 + 高维稀疏 → 引出词向量(11.3)
实战: sklearn CountVectorizer/TfidfVectorizer; TF-IDF+线性模型至今是强基线
```

### 💡 面试速查 / Interview cheat-sheet
1. **TF-IDF**: TF×IDF; IDF=log(N/df) 压低普遍词、突出独特词=关键词。
   TF-IDF: TF×IDF; IDF=log(N/df) suppresses common, highlights distinctive = keywords.
2. **为什么要 IDF**: 纯词频被 the/is 主导, 没区分度。
   Why IDF: raw counts dominated by the/is, no discrimination.
3. **n-gram**: 捕捉词序/否定(not good); 代价维度暴涨。
   n-grams: capture order/negation; cost: dimensionality explosion.
4. **稀疏高维**: 维度=词表大小, 非零极少。
   Sparse & high-dim: dimension = vocab size, very few nonzeros.
5. **局限**: 无语义+无词序 → 词向量(Word2Vec)来解决。
   Limits: no semantics/order → solved by word embeddings.

### 下一节 / Next
**11.3 词向量**——BoW/TF-IDF 的词彼此正交、不含语义。词向量(Word2Vec)用**低维稠密**向量让"语义相近的词在空间里也相近"，还能做"国王−男人+女人≈女王"的类比。我们会**从零实现 Skip-gram + 负采样**。
**11.3 Word Embeddings** — BoW/TF-IDF words are orthogonal and semantics-free. Word embeddings (Word2Vec) use **low-dimensional dense** vectors so "similar-meaning words are close," even enabling "king − man + woman ≈ queen." We'll **implement Skip-gram + negative sampling from scratch**.
